# Tutorial 2: DeepE2EROM Models Module tutorial

This tutorial demonstrates how to create end-to-end reduced order models with the DeepE2EROM models module.

## Table of Contents
1. [Overview](#overview)
2. [Basic Model Creation](#basic-creation)
3. [Architecture Specifications](#architecture-specs)
4. [Dynamics Options](#dynamics-options)
5. [Model Inference](#inference)
5. [Control Autoencoder](#input-autoencoder)
6. [Pre-built Components](#prebuilt-components)
7. [Custom Dynamics](#custom-dynamics)

## 1. Overview <a name="overview"></a>

The DeepE2EROM models module provides flexible creation of end-to-end reduced order models with three main components:

- **State Autoencoder**: Encodes high-dimensional states to low-dimensional latent space
- **Dynamics**: Predicts latent state evolution (linear, control-affine, LSTM, or custom)
- **Control Autoencoder** (optional): Encodes control variables

Key features:
- Flexible architecture specifications
- Multiple dynamics options
- Support for 1D and 2D data

### End-to-End ROM

The DeepE2EROM processes input sequences of high-dimensional states with shape `(batch_size, lookback, *state_dim)` and control inputs with shape `(batch_size, lookback, control_dim)` to predict subsequent system states. For multi-step prediction, the model requires in addition to the `lookback` control inputs the future control inputs, which makes the shape of the control sequence as `(batch_size, looback + pred_horizon - 1, control_dim)`.  If provided with a control sequence longer than needed, the model automatically truncates it to the required length, ignoring any excess control inputs. Internally, the state encoder compresses the input state sequence to latent representations of shape `(batch_size, lookback, latent_dim)`, while an optional control encoder similarly processes control vectors when enabled. The core dynamics module then operates exclusively in the latent space, processing sequences of latent states and controls to predict future latent states.

All dynamical models implemented in DeepE2EROM are designed to process these latent sequences, ensuring consistent interface while allowing specialized dynamics formulations. The decoder finally reconstructs the predicted latent states back to the original high-dimensional space, completing the end-to-end prediction pipeline.

---

## 2. Basic Model Creation <a name="basic-creation"></a>

Let's start with imports and a basic model configuration.

In [1]:
import deepe2erom
import torch

print("DeepE2Erom version: " + deepe2erom.__version__)

# Basic configuration for a 1D system
state_dim = 64  # 1D state with 64 grid points
control_dim = 2
lookback = 10
pred_horizon = 5
latent_dim = 8

print("\nBasic Configuration:")
print(f"  State dimension: {state_dim}")
print(f"  Control dimension: {control_dim}")
print(f"  Lookback steps: {lookback}")
print(f"  Prediction horizon: {pred_horizon}")
print(f"  Latent dimension: {latent_dim}")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Using device: {device}")

DeepE2Erom version: 0.1.0

Basic Configuration:
  State dimension: 64
  Control dimension: 2
  Lookback steps: 10
  Prediction horizon: 5
  Latent dimension: 8
  Using device: cuda


DeepE2EROM provides an easy and structured way to create ready-to-train ROMs using the `create_model` function. This function handles all the complexity of connecting components while providing maximum flexibility.

### Understanding the `create_model` Function

The `create_model` function is the main entry point for creating DeepE2EROM models. Below we break down all its arguments:

#### Required Arguments:
- `state_dim`: Dimension of the state space (int for 1D, tuple for 2D/3D)
- `control_dim`: Dimension of control inputs
- `lookback`: Number of past time steps used for prediction context
- `pred_horizon`: Number of future time steps to predict
- `latent_dim`: Dimension of the latent space

#### Autoencoder Components:

**State Autoencoder (choose one per component):**
- `encoder_arch` OR `encoder`: Architecture specification or pre-built Pytorch `nn.Module`
- `decoder_arch` OR `decoder`: Architecture specification or pre-built Pytorch `nn.Module`

**Control Autoencoder (optional):**
- `use_control_autoencoder`: Enable encoding of control vector
- `control_latent_dim`: Latent dimension for encoded controls
- `control_encoder_arch` OR `control_encoder`: Architecture specification or pre-built Pytorch `nn.Module`
- `control_decoder_arch` OR `control_decoder`: Architecture specification or pre-built Pytorch `nn.Module`

#### Dynamics Options (choose one approach):

**Option A: Built-in Dynamics Types**

DeepE2EROM supports the creation of the following latent dynamic models defined by the variable `dynamics_type`:
- `dynamics_type`: "control_affine", "lstm", or "linear"
- For `control_affine`: The user must provide `drift_arch` and `input_arch` for architecture specification of the drift and input networks
- For `lstm`: Optionally set `lstm_hidden_size` and `lstm_layers` (Default values are set to 128 and 2, respectively)

**Option B: Custom Dynamics**
- `dynamics`: Pre-built PyTorch module implementing custom dynamics

---

## 3. Architecture Specifications <a name="architecture-specs"></a>

### Available Layer Types

The architecture specification supports these layer types:

- `Linear`: Fully connected layers
- `Conv2d`, `ConvTranspose2d`: Convolutional layers (for 2D data)
- `ReLU`, `Sigmoid`, `Tanh`: Activation functions
- `Flatten`, `Unflatten`: Shape manipulation
- `BatchNorm1d`, `BatchNorm2d`: Normalization
- `Dropout`: Regularization

Each layer takes a `params` dictionary with appropriate parameters.

In [2]:
# Define architectures for encoder and decoder
encoder_arch = [
    {"type": "Linear", "params": {"in_features": 64, "out_features": 32}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 32, "out_features": 8}}
]

decoder_arch = [
    {"type": "Linear", "params": {"in_features": 8, "out_features": 32}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 32, "out_features": 64}}
]

In this manner, we can easily create autoencoder architectures to process 2D data as well. 

In [3]:
encoder_arch_2D = [
    {"type": "Conv2d", "params": {"in_channels": 1, "out_channels": 4, "kernel_size": 3, "stride": 2, "padding": 1}},
    {"type": "ReLU"},
    {"type": "Conv2d", "params": {"in_channels": 4, "out_channels": 8, "kernel_size": 3, "stride": 2, "padding": 1}},
    {"type": "ReLU"},
    {"type": "Conv2d", "params": {"in_channels": 8, "out_channels": 16, "kernel_size": 3, "stride": 2, "padding": 1}},
    {"type": "ReLU"},
    {"type": "Conv2d", "params": {"in_channels": 16, "out_channels": 32, "kernel_size": 3, "stride": 2, "padding": 1}},
    {"type": "ReLU"},
    {"type": "Flatten"},
    {"type": "Linear", "params": {"in_features": 32 * 4 * 4, "out_features": 128}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 128, "out_features": latent_dim}}
]

decoder_arch_2D = [
    {"type": "Linear", "params": {"in_features": latent_dim, "out_features": 128}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 128, "out_features": 32 * 4 * 4}},
    {"type": "ReLU"},
    {"type": "Unflatten", "params": {"dim": 1, "unflattened_size": (32, 4, 4)}},
    {"type": "ConvTranspose2d", "params": {"in_channels": 32, "out_channels": 16, "kernel_size": 3, "stride": 2, "padding": 1, "output_padding": 1}},
    {"type": "ReLU"},
    {"type": "ConvTranspose2d", "params": {"in_channels": 16, "out_channels": 8, "kernel_size": 3, "stride": 2, "padding": 1, "output_padding": 1}},
    {"type": "ReLU"},
    {"type": "ConvTranspose2d", "params": {"in_channels": 8, "out_channels": 4, "kernel_size": 3, "stride": 2, "padding": 1, "output_padding": 1}},
    {"type": "ReLU"},
    {"type": "ConvTranspose2d", "params": {"in_channels": 4, "out_channels": 1, "kernel_size": 3, "stride": 2, "padding": 1, "output_padding": 1}},
    {"type": "Sigmoid"}
]

---

## 4. Dynamics Options <a name="dynamics-options"></a>

DeepE2EROM supports multiple dynamics types: linear, control-affine, and LSTM. Given a state latent sequence `Z` of shape `(batch_size, lookback, latent_dim)` and (latent) control sequence `U` of shape `(batch_size, lookback, control_dim)`, the dynamical model aims to predict the suqsequence latent vector as `next_latent = dynamics(Z, U)`.

### Creating a Model with Linear Dynamics


The linear dynamics module implements a straightforward approach for latent state prediction as `next_latent = A · ksi + B · u_current`, where
- **ksi**: Extended feature vector created by flattening and concatenating:
  - All `lookback` latent state vectors
  - The first `lookback - 1` control input vectors
  - Resulting dimension: `latent_dim × lookback + control_dim × (lookback - 1)`

- **u_current**: Current control input (the last element in the control sequence)

- **A**: Learnable weight matrix mapping the extended feature vector to latent space
  - Dimensions: `(latent_dim, latent_dim × lookback + control_dim × (lookback - 1))`

- **B**: Learnable weight matrix mapping the current control to latent space
  - Dimensions: `(latent_dim, control_dim)`

In [4]:
from deepe2erom import create_model

# Create model with linear dynamics (device is automatically set)
model_linear = create_model(
    state_dim=state_dim,
    control_dim=control_dim,
    lookback=lookback,
    pred_horizon=pred_horizon,
    latent_dim=latent_dim,
    dynamics_type="linear",
    encoder_arch=encoder_arch,
    decoder_arch=decoder_arch,
    device=device # the default is cpu if not specified
)

print("Model with Linear Dynamics Created:")
print(f"  Encoder: {model_linear.encoder}")
print(f"  Decoder: {model_linear.decoder}")
print(f"  Dynamics: {model_linear.dynamics}")
print(f"  Device: {model_linear.config.device}")

Model with Linear Dynamics Created:
  Encoder: Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=8, bias=True)
)
  Decoder: Sequential(
  (0): Linear(in_features=8, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=64, bias=True)
)
  Dynamics: LinearDynamics(
  (A): Linear(in_features=98, out_features=8, bias=True)
  (B): Linear(in_features=2, out_features=8, bias=True)
)
  Device: cuda


In [5]:
# Count parameters
param_counts = model_linear.count_parameters(verbose=True)

MODEL PARAMETER COUNT
encoder         |    2,344 total |    2,344 trainable
decoder         |    2,400 total |    2,400 trainable
dynamics        |      816 total |      816 trainable
------------------------------------------------------------
TOTAL           |    5,560 total |    5,560 trainable


### Creating a Model with Control-Affine Dynamics

The control-affine dynamics module implements a physically-motivated approach for latent state prediction that captures nonlinear system behavior while maintaining an affine relationship with control inputs. Given input sequences of latent states `Z` with shape `(batch_size, lookback, latent_dim)` and control inputs `U` with shape `(batch_size, lookback, control_dim)`, the module predicts the next latent state through the operation: `next_latent = drift_net(ksi) + input_net(ksi) * u_current`, where:

- **ksi**: Extended feature vector formed as mentioned above

- **u_current**: Current control input (last element of the control sequence)

- **drift_net**: Neural network modeling the autonomous system dynamics
  - Maps `ksi` to latent space: `(latent_dim)`
  - Captures intrinsic system evolution without external control

- **input_net**: Neural network modeling the control influence matrix
  - Maps `ksi` to a matrix: `(latent_dim × control_dim)`
  - Output reshaped to form the control effectiveness matrix


When creating control-affine latent dynamics, the user has to define the architecture of the drift and input networks (or provide them as Pytorch `nn.Module`). The user should carefully design the input and output layers of these networks. Specifically:
- The dimension of the input layers should match the dimension of `ksi`.
- The dimension of the output layer of the drift network should be `latent_dim`.
- The dimension of the output layer of the input network should be `latent_dim*control_dim` or `latent_dim*input_latent_dim` when using a control autoencoder.

In [6]:
# Define architectures for drift and input networks
ksi_dim = latent_dim * lookback + control_dim * (lookback - 1) 
out_dim = latent_dim * control_dim 

drift_arch = [
    {"type": "Linear", "params": {"in_features": ksi_dim, "out_features": 64}}, 
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 64, "out_features": latent_dim}}
]

input_arch = [
    {"type": "Linear", "params": {"in_features": ksi_dim, "out_features": 64}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 64, "out_features": out_dim}} 
]

# Create model with control-affine dynamics
model_control_affine = create_model(
    state_dim=state_dim,
    control_dim=control_dim,
    lookback=lookback,
    pred_horizon=pred_horizon,
    latent_dim=latent_dim,
    dynamics_type="control_affine",
    drift_arch=drift_arch,
    input_arch=input_arch,
    encoder_arch=encoder_arch,
    decoder_arch=decoder_arch,
    device=device
)

print("Model with Control-Affine Dynamics Created:")
print(f"  Encoder: {model_control_affine.encoder}")
print(f"  Decoder: {model_control_affine.decoder}")
print(f"  Dynamics: {model_control_affine.dynamics}")
print(f"  Device: {model_control_affine.config.device}")

Model with Control-Affine Dynamics Created:
  Encoder: Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=8, bias=True)
)
  Decoder: Sequential(
  (0): Linear(in_features=8, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=64, bias=True)
)
  Dynamics: ControlAffineDynamics(
  (drift_net): Sequential(
    (0): Linear(in_features=98, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=8, bias=True)
  )
  (input_net): Sequential(
    (0): Linear(in_features=98, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
  )
)
  Device: cuda


In [7]:
# Count parameters
param_counts = model_control_affine.count_parameters(verbose=True)

MODEL PARAMETER COUNT
encoder         |    2,344 total |    2,344 trainable
decoder         |    2,400 total |    2,400 trainable
dynamics        |   14,232 total |   14,232 trainable
------------------------------------------------------------
TOTAL           |   18,976 total |   18,976 trainable


In case of providing an architecture that does not fit the creation of the control-affine model given above, `create_model` will correct the dimensions of the input and output layers while preserving the remaining architecture.

In [8]:
drift_arch_wrong = [
    {"type": "Linear", "params": {"in_features": 1, "out_features": 64}}, 
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 64, "out_features": 1}}
]

input_arch_wrong = [
    {"type": "Linear", "params": {"in_features": 1, "out_features": 64}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 64, "out_features": 1}} 
]

# Create model with control-affine dynamics
model_control_affine_corrected = create_model(
    state_dim=state_dim,
    control_dim=control_dim,
    lookback=lookback,
    pred_horizon=pred_horizon,
    latent_dim=latent_dim,
    dynamics_type="control_affine",
    drift_arch=drift_arch_wrong,
    input_arch=input_arch_wrong,
    encoder_arch=encoder_arch,
    decoder_arch=decoder_arch,
    device=device
)

print("Model with Control-Affine Dynamics Created:")
print(f"  Encoder: {model_control_affine_corrected.encoder}")
print(f"  Decoder: {model_control_affine_corrected.decoder}")
print(f"  Dynamics: {model_control_affine_corrected.dynamics}")

Model with Control-Affine Dynamics Created:
  Encoder: Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=8, bias=True)
)
  Decoder: Sequential(
  (0): Linear(in_features=8, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=64, bias=True)
)
  Dynamics: ControlAffineDynamics(
  (drift_net): Sequential(
    (0): Linear(in_features=98, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=8, bias=True)
  )
  (input_net): Sequential(
    (0): Linear(in_features=98, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
  )
)


/home/users/amj/Repos/deepe2erom/src/deepe2erom/models/dynamics.py:66: UserWarning: drift_arch: First layer in_features 1 does not match expected 98. Correcting to 98.
  warnings.warn(f"{arch_name}: First layer in_features {adjusted_arch[0]['params']['in_features']} does not match expected {expected_in}. Correcting to {expected_in}.")
/home/users/amj/Repos/deepe2erom/src/deepe2erom/models/dynamics.py:72: UserWarning: drift_arch: Last layer out_features 1 does not match expected 8. Correcting to 8.
  warnings.warn(f"{arch_name}: Last layer out_features {adjusted_arch[-1]['params']['out_features']} does not match expected {expected_out}. Correcting to {expected_out}.")
/home/users/amj/Repos/deepe2erom/src/deepe2erom/models/dynamics.py:66: UserWarning: input_arch: First layer in_features 1 does not match expected 98. Correcting to 98.
  warnings.warn(f"{arch_name}: First layer in_features {adjusted_arch[0]['params']['in_features']} does not match expected {expected_in}. Correcting to {exp

### Creating a Model with LSTM Dynamics

The LSTM dynamics module leverages recurrent neural networks to capture complex temporal dependencies in latent state evolution. Given input sequences of latent states `Z` with shape `(batch_size, lookback, latent_dim)` and control inputs `U` with shape `(batch_size, lookback, control_dim)`, the module predicts the next latent state through sophisticated sequence processing.

In [9]:
# Create model with LSTM dynamics
model_lstm = create_model(
    state_dim=state_dim,
    control_dim=control_dim,
    lookback=lookback,
    pred_horizon=pred_horizon,
    latent_dim=latent_dim,
    dynamics_type="lstm",
    lstm_hidden_size=32,
    lstm_layers=2,
    encoder_arch=encoder_arch,
    decoder_arch=decoder_arch,
    device=device
)

print("Model with LSTM Dynamics Created:")
print(f"  Encoder: {model_lstm.encoder}")
print(f"  Decoder: {model_lstm.decoder}")
print(f"  Dynamics: {model_lstm.dynamics}")
print(f"  Device: {next(model_lstm.parameters()).device}")

Model with LSTM Dynamics Created:
  Encoder: Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=8, bias=True)
)
  Decoder: Sequential(
  (0): Linear(in_features=8, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=64, bias=True)
)
  Dynamics: LSTMcDynamics(
  (lstm): LSTM(10, 32, num_layers=2, batch_first=True)
  (output_net): Linear(in_features=32, out_features=8, bias=True)
)
  Device: cuda:0


In [10]:
# parameter count
param_counts = model_lstm.count_parameters(verbose=True)

MODEL PARAMETER COUNT
encoder         |    2,344 total |    2,344 trainable
decoder         |    2,400 total |    2,400 trainable
dynamics        |   14,344 total |   14,344 trainable
------------------------------------------------------------
TOTAL           |   19,088 total |   19,088 trainable


## 5. Model Inference <a name="inference"></a>

Every forward pass of DeepE2EROM performs multi-step prediction of future high-dimensional states over the specified `pred_horizon`. The prediction is conducted autoregressively within the latent space, where each predicted latent state becomes part of the input for subsequent predictions.

The forward method returns a tuple of outputs depending on whether an input autoencoder is used:

- **decoded_sequence**: The reconstructed input state sequence from the autoencoder, with shape `(batch_size, lookback, *state_dims)`. This represents the model's reconstruction of the provided input states.
- **pred_latents**: The predicted latent state sequence over the prediction horizon, with shape `(batch_size, pred_horizon, latent_dim)`. These are the latent representations of the predicted future states.
- **pred_states**: The predicted high-dimensional state sequence over the prediction horizon, with shape `(batch_size, pred_horizon, *state_dims)`. These are the decoded predictions in the original state space.

If the model uses an input autoencoder (`use_input_autoencoder=True`), an additional output is included:
- **decoded_input_sequence**: The reconstructed input control sequence from the input autoencoder. This represents the model's reconstruction of the provided control inputs.

The autoregressive prediction process ensures that each step's predicted latent state is fed back into the dynamics for the next prediction, allowing for multi-step forecasting without external ground truth.

In [11]:
import torch

# sample input data
batch_size = 1
x = torch.randn(batch_size, lookback, state_dim).to(device)
u = torch.randn(batch_size, lookback+pred_horizon, control_dim).to(device)

# Forward pass - multi-step prediction (all models have the same interface)
with torch.no_grad():
    #decoded_seq, pred_latents, pred_states = model_linear(x, u)
    decoded_seq, pred_latents, pred_states = model_control_affine(x, u)
    #decoded_seq, pred_latents, pred_states = model_lstm(x, u)

print("Decoded sequence shape: ", decoded_seq.shape)
print("Predicted latent sequence shape: ", pred_latents.shape)
print("Predicted high-dimensional sequence shape: ", pred_states.shape)

Decoded sequence shape:  torch.Size([1, 10, 64])
Predicted latent sequence shape:  torch.Size([1, 5, 8])
Predicted high-dimensional sequence shape:  torch.Size([1, 5, 64])


The length of the prediction horizon can be adjusted dynamically for inference, allowing flexible multi-step forecasting. This is useful for scenarios where you want to predict further into the future or limit predictions for faster inference.

Note that the control sequence length must accommodate the new horizon (i.e., the control sequence length should be at least `lookback + pred_horizon - 1`).

In [12]:
# predict single step
model_lstm.config.pred_horizon = 1

# Forward pass
with torch.no_grad():
    decoded_seq, pred_latents, pred_states = model_lstm(x, u)

print("Decoded sequence shape: ", decoded_seq.shape)
print("Predicted latent sequence shape: ", pred_latents.shape)
print("Predicted high-dimensional sequence shape: ", pred_states.shape)

Decoded sequence shape:  torch.Size([1, 10, 64])
Predicted latent sequence shape:  torch.Size([1, 1, 8])
Predicted high-dimensional sequence shape:  torch.Size([1, 1, 64])


### Simulation

Alternatively, the method `simulate` can be used to generate long-term predictions given an initial condition sequence and a future control sequence. This method handles the autoregressive loop internally and temporarily adjusts the prediction horizon.

The `simulate` method accepts both **Numpy arrays** and **PyTorch tensors** (which are automatically moved to the correct device) with the following unbatched shapes:

- **x0** (Initial Condition): `(lookback, *state_dims)` - The sequence of past states required to initialize the model.
- **u** (Control Sequence): `(sim_horizon + lookback - 1, control_dim)` - The control inputs covering the initial lookback period and the entire simulation horizon.
- **sim_horizon** (int): The number of future time steps to simulate.

It returns the predicted latent and state trajectories as CPU tensors.

In [13]:
# Define simulation parameters
sim_horizon = 20

# Create initial condition (unbatched)
x0 = torch.randn(lookback, state_dim)

# Create control sequence for the full simulation
# Length must be: lookback + sim_horizon - 1
u_sim = torch.randn(lookback + sim_horizon - 1, control_dim)

print(f"Simulation inputs:")
print(f"  x0 shape: {x0.shape}")
print(f"  u shape: {u_sim.shape}")
print(f"  Horizon: {sim_horizon}")

# Run simulation
pred_latents_sim, pred_states_sim = model_lstm.simulate(x0, u_sim, sim_horizon)

print("\nSimulation outputs:")
print(f"  Predicted latents shape: {pred_latents_sim.shape}") # (sim_horizon, latent_dim)
print(f"  Predicted states shape: {pred_states_sim.shape}")   # (sim_horizon, state_dim)

Simulation inputs:
  x0 shape: torch.Size([10, 64])
  u shape: torch.Size([29, 2])
  Horizon: 20

Simulation outputs:
  Predicted latents shape: torch.Size([20, 8])
  Predicted states shape: torch.Size([20, 64])


---

## 6. Control Autoencoder <a name="control-autoencoder"></a>

For high-dimensional controls, one can use an input autoencoder to reduce control dimensionality. The `create_model` function handles the creation of the control autoencoder by setting the `use_control_autoencoder` to `True` and specifiying the required architectures or providing the models.

Below is an example for creating an End-to-End control-affine ROM for a 2D field and using a control autoencoder.

In [14]:
# Basic configuration for a 2D system with high-dimensional control vector
state_dim = (1,64,64) # single channel 64x64 frames
control_dim = 100
lookback = 5
pred_horizon = 3
latent_dim = 8
control_latent_dim = 4

# Define input autoencoder architectures
control_encoder_arch = [
    {"type": "Linear", "params": {"in_features": control_dim, "out_features": 64}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 64, "out_features": control_latent_dim}},
]

control_decoder_arch = [
    {"type": "Linear", "params": {"in_features": control_latent_dim, "out_features": 64}},
    {"type": "ReLU"},
    {"type": "Linear", "params": {"in_features": 64, "out_features": control_dim}},
]  

# Create model with input autoencoder
model_with_input_ae = create_model(
    state_dim=state_dim,
    control_dim=control_dim,
    lookback=lookback,
    pred_horizon=pred_horizon,
    latent_dim=latent_dim,
    control_latent_dim=control_latent_dim,
    dynamics_type="control_affine",
    drift_arch=drift_arch, # the input/output dimensions will be automatically corrected
    input_arch=input_arch, # the input/output dimensions will be automatically corrected
    use_control_autoencoder=True,
    encoder_arch=encoder_arch_2D,
    decoder_arch=decoder_arch_2D,
    control_encoder_arch=control_encoder_arch,
    control_decoder_arch=control_decoder_arch,
    device=device
)

print("Model with Input Autoencoder:")
print(f"  State Encoder: {model_with_input_ae.encoder}")
print(f"  State Decoder: {model_with_input_ae.decoder}")
print(f"  Control Encoder: {model_with_input_ae.control_encoder}")
print(f"  Control Decoder: {model_with_input_ae.control_decoder}")
print(f"  Device: {model_with_input_ae.config.device}")

Model with Input Autoencoder:
  State Encoder: Sequential(
  (0): Conv2d(1, 4, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (1): ReLU()
  (2): Conv2d(4, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (3): ReLU()
  (4): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (5): ReLU()
  (6): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (7): ReLU()
  (8): Flatten(start_dim=1, end_dim=-1)
  (9): Linear(in_features=512, out_features=128, bias=True)
  (10): ReLU()
  (11): Linear(in_features=128, out_features=8, bias=True)
)
  State Decoder: Sequential(
  (0): Linear(in_features=8, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=512, bias=True)
  (3): ReLU()
  (4): Unflatten(dim=1, unflattened_size=(32, 4, 4))
  (5): ConvTranspose2d(32, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
  (6): ReLU()
  (7): ConvTranspose2d(16, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1

/home/users/amj/Repos/deepe2erom/src/deepe2erom/models/dynamics.py:66: UserWarning: drift_arch: First layer in_features 98 does not match expected 56. Correcting to 56.
  warnings.warn(f"{arch_name}: First layer in_features {adjusted_arch[0]['params']['in_features']} does not match expected {expected_in}. Correcting to {expected_in}.")
/home/users/amj/Repos/deepe2erom/src/deepe2erom/models/dynamics.py:66: UserWarning: input_arch: First layer in_features 98 does not match expected 56. Correcting to 56.
  warnings.warn(f"{arch_name}: First layer in_features {adjusted_arch[0]['params']['in_features']} does not match expected {expected_in}. Correcting to {expected_in}.")
/home/users/amj/Repos/deepe2erom/src/deepe2erom/models/dynamics.py:72: UserWarning: input_arch: Last layer out_features 16 does not match expected 32. Correcting to 32.
  warnings.warn(f"{arch_name}: Last layer out_features {adjusted_arch[-1]['params']['out_features']} does not match expected {expected_out}. Correcting to 

---

## 7. Pre-built Components <a name="prebuilt-components"></a>

While architecture specifications provide flexibility and ease of configuration, you can also pass pre-built PyTorch `nn.Module` instances directly to `create_model`. This is particularly useful when:

- You have custom or pre-trained components
- You want to reuse models from other projects
- You need more complex architectures not easily defined by the specification format
- You prefer programmatic construction over dictionary-based specs

Pre-built components offer full control over initialization, custom layers, and advanced features like weight initialization or custom forward passes. However, they require manual dimension management to ensure compatibility with the model's configuration.

### Using Pre-built Encoders and Decoders

Here's how to use pre-built encoder and decoder modules:

In [15]:
import torch.nn as nn

# Pre-built encoder and decoder with custom initialization
encoder = nn.Sequential(
    nn.Linear(64, 32),
    nn.BatchNorm1d(32),  # Added normalization for better training
    nn.ReLU(),
    nn.Dropout(0.1),     # Added regularization
    nn.Linear(32, 8)
)

decoder = nn.Sequential(
    nn.Linear(8, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(32, 64)
)

# Create model with pre-built components
model_prebuilt = create_model(
    state_dim=64,
    control_dim=2,
    lookback=10,
    pred_horizon=5,
    latent_dim=8,
    dynamics_type="linear",
    encoder=encoder,
    decoder=decoder,
    device=device
)

print("Model with Pre-built Components:")
print(f"  Encoder: {model_prebuilt.encoder}")
print(f"  Decoder: {model_prebuilt.decoder}")
print(f"  Dynamics: {model_prebuilt.dynamics}")

Model with Pre-built Components:
  Encoder: Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Dropout(p=0.1, inplace=False)
  (4): Linear(in_features=32, out_features=8, bias=True)
)
  Decoder: Sequential(
  (0): Linear(in_features=8, out_features=32, bias=True)
  (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Dropout(p=0.1, inplace=False)
  (4): Linear(in_features=32, out_features=64, bias=True)
)
  Dynamics: LinearDynamics(
  (A): Linear(in_features=98, out_features=8, bias=True)
  (B): Linear(in_features=2, out_features=8, bias=True)
)


---

## 8. Custom Dynamics <a name="custom-dynamics"></a>

DeepE2EROM allows you to define custom dynamics modules by inheriting from `nn.Module` and implementing a `forward` method that adheres to a specific interface. This enables integration of specialized dynamics models, such as physics-informed neural networks, symbolic regression-based models, or any custom architecture.

### Interface Requirements

Custom dynamics modules must implement the following interface:

- **Input Shapes**:
  - `latent_window`: Tensor of shape `(batch_size, lookback, latent_dim)` - sequence of latent states
  - `control_window`: Tensor of shape `(batch_size, lookback, control_dim)` or `(batch_size, lookback, control_latent_dim)` if using control autoencoder - sequence of controls

- **Output Shape**:
  - `next_latent`: Tensor of shape `(batch_size, latent_dim)` - predicted next latent state

- **Key Points**:
  - The module should process the full `latent_window` and `control_window` to make predictions
  - Maintain the common interface to ensure compatibility with the rest of the DeepE2EROM framework
  - The `config` object provides access to model parameters like `latent_dim`, `lookback`, etc.

### Example: Simple Feedforward Custom Dynamics

Here's an example of a custom dynamics module that uses a simple feedforward network:

In [16]:
class CustomDynamics(nn.Module):
    """Example custom dynamics: Simple feedforward network."""
    
    def __init__(self, latent_dim: int, lookback: int, control_dim: int):
        super().__init__()
        self.latent_dim = latent_dim
        self.lookback = lookback
        self.control_dim = control_dim
        input_dim = latent_dim * lookback + self.control_dim * (lookback - 1)
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
    
    def forward(self, latent_window, control_window):
        # Flatten the latent and control windows
        latent_flat = latent_window.view(latent_window.size(0), -1)  # (batch, lookback * latent_dim)
        control_flat = control_window[:, :-1].view(control_window.size(0), -1)  # (batch, (lookback-1) * control_dim)
        
        # Concatenate for input to network
        combined = torch.cat([latent_flat, control_flat], dim=-1)
        
        # Predict next latent state
        return self.net(combined)

# Create custom dynamics instance with explicit parameters
state_dim = 64
lookback = 10
pred_horizon = 7
latent_dim = 8
control_dim = 2
custom_dynamics = CustomDynamics(
    latent_dim=latent_dim,
    lookback=lookback,
    control_dim=control_dim,
)

# Create model with custom dynamics
model_custom = create_model(
    state_dim=state_dim,
    control_dim=control_dim,
    lookback=lookback,
    pred_horizon=pred_horizon,
    latent_dim=latent_dim,
    dynamics=custom_dynamics,  # Pass pre-built custom dynamics
    encoder_arch=encoder_arch,
    decoder_arch=decoder_arch
)

print(f"  Dynamics: {type(model_custom.dynamics).__name__}")
print(f"  Custom Network: {model_custom.dynamics}")

  Dynamics: CustomDynamics
  Custom Network: CustomDynamics(
  (net): Sequential(
    (0): Linear(in_features=98, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=8, bias=True)
  )
)


In [19]:
# Inference with custom dynamics (same interface as other models)
pred_horizon = 100
x = torch.randn(lookback, state_dim)
u = torch.randn(lookback + pred_horizon, control_dim)

# simulate
pred_latents, pred_states = model_custom.simulate(x, u, pred_horizon)

print("Custom Dynamics Inference:")
print("Predicted latent sequence shape: ", pred_latents.shape)
print("Predicted high-dimensional sequence shape: ", pred_states.shape)

Custom Dynamics Inference:
Predicted latent sequence shape:  torch.Size([100, 8])
Predicted high-dimensional sequence shape:  torch.Size([100, 64])


By following this interface, custom dynamics modules seamlessly integrate with DeepE2EROM's training and inference pipelines, maintaining the common API across different model types.